# Academic Admission Classification
Binary classification of graduate admission from academic scores.

## Project Overview
Classify whether a student will be admitted based on GRE, TOEFL, CGPA, and other academic factors, by binarizing the admission probability.

## Learning Objectives
- Binarize a continuous target for classification
- Handle small, clean tabular datasets
- Compare boosting classifiers on academic data

## Problem Statement
Given academic metrics, predict whether a student's admission chance is above 0.5 (admitted) or below (not admitted).

## Why This Project Matters
Admission prediction helps students prioritize applications and universities assess applicant pools efficiently.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | Kaggle: mohansacharya/graduate-admissions |
| **Target** | Admission (binary: chance >= 0.5) |
| **Rows** | ~500 |
| **Features** | GRE, TOEFL, CGPA, University Rating, SOP, LOR, Research |

## Dataset Source & License
Kaggle: mohansacharya/graduate-admissions. Educational dataset, public license.

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

## Imports

In [ ]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [ ]:
THRESHOLD = 0.5  # Binarize admission chance at this threshold

## Dataset Loading

In [ ]:
import kagglehub
path = kagglehub.dataset_download('mohansacharya/graduate-admissions')
import glob
csvs = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
print('Files:', csvs)
df = pd.read_csv(csvs[0])
df.columns = df.columns.str.strip()
print(f'Shape: {df.shape}')
df.head()

## Data Validation

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

# Find admission chance column
chance_cols = [c for c in df.columns if 'admit' in c.lower() or 'chance' in c.lower()]
CHANCE_COL = chance_cols[0] if chance_cols else df.select_dtypes('number').columns[-1]
print(f'\nChance column: {CHANCE_COL}')

# Binarize
df['Admitted'] = (df[CHANCE_COL] >= THRESHOLD).astype(int)
TARGET = 'Admitted'
print(f'\nClass distribution:')
print(df[TARGET].value_counts())

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df[TARGET].value_counts().plot.bar(ax=axes[0,0], color=['#e74c3c','#2ecc71'], edgecolor='black')
axes[0,0].set_title('Admitted vs Not')
axes[0,0].set_xticklabels(['Not Admitted','Admitted'], rotation=0)

for i, col in enumerate(['GRE Score', 'TOEFL Score', 'CGPA']):
    if col in df.columns:
        ax = axes[(i+1)//2, (i+1)%2]
        for label in [0, 1]:
            subset = df[df[TARGET] == label]
            ax.hist(subset[col], bins=20, alpha=0.5, label=f'{"Adm" if label else "Not"}', edgecolor='black')
        ax.set_title(col)
        ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [ ]:
print(df[TARGET].value_counts())
print(f'\nImbalance ratio: {df[TARGET].value_counts()[0] / df[TARGET].value_counts()[1]:.2f}')

## Train/Test Split

In [ ]:
# Drop serial number and original chance column
drop_cols = [c for c in df.columns if 'serial' in c.lower() or c.lower() == 'no']
drop_cols.append(CHANCE_COL)
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

X = df.drop(columns=[TARGET])
y = df[TARGET]
X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train class dist: {y_train.value_counts().to_dict()}')

## Preprocessing
Dropped serial number and original probability column. All features numeric.

## Feature Engineering
Using raw scores directly. Small dataset — avoiding overfitting with minimal engineering.

## Baseline Model

In [ ]:
bl = LogisticRegression(max_iter=1000, random_state=SEED)
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline LR: Acc={accuracy_score(y_test, bl_pred):.4f}  F1={f1_score(y_test, bl_pred):.4f}')
print(classification_report(y_test, bl_pred))

## LazyPredict Benchmark

In [ ]:
# --- LazyPredict Benchmark ---
try:
    from lazypredict.Supervised import LazyClassifier
    n_lp = min(5000, len(X_train))
    lc = LazyClassifier(verbose=0, ignore_warnings=True)
    lp_models, _ = lc.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

## FLAML AutoML

In [ ]:
# --- FLAML AutoML ---
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='classification', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  Accuracy={accuracy_score(y_test, flaml_pred):.4f}  F1={f1_score(y_test, flaml_pred, average='weighted'):.4f}")
    if hasattr(automl, 'predict_proba'):
        try:
            flaml_proba = automl.predict_proba(X_test)
            if flaml_proba.shape[1] == 2:
                print(f"  ROC-AUC={roc_auc_score(y_test, flaml_proba[:, 1]):.4f}")
            else:
                print(f"  ROC-AUC={roc_auc_score(y_test, flaml_proba, multi_class='ovr', average='weighted'):.4f}")
        except: pass
except Exception as e:
    print(f"FLAML skipped: {e}")

## Additional Models: CatBoost, LightGBM, XGBoost

In [ ]:
# --- Boosting Models ---
models = {}
# CatBoost
cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

# LightGBM
lgb = LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

# XGBoost
xgb = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
models['XGBoost'] = xgb

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: Acc={accuracy_score(y_test, p):.4f}  F1={f1_score(y_test, p, average='weighted'):.4f}")

## Top 3 Model Selection

In [ ]:
# --- Top 3 Selection ---
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }

results_df = pd.DataFrame(all_results).T.sort_values('F1', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

## Final Evaluation of Top 3

In [ ]:
# --- Final Evaluation of Top 3 ---
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    final_results[name] = {
        'Accuracy': accuracy_score(y_test, p),
        'F1': f1_score(y_test, p, average='weighted'),
        'Precision': precision_score(y_test, p, average='weighted'),
        'Recall': recall_score(y_test, p, average='weighted'),
    }
    print(f"{name}: Acc={final_results[name]['Accuracy']:.4f}  F1={final_results[name]['F1']:.4f}")
    print(classification_report(y_test, p))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(final_results.keys())
for i, metric in enumerate(['Accuracy', 'F1']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

## Error Analysis

In [ ]:
# --- Error Analysis ---
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)

from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=axes[0], cmap='Blues')
axes[0].set_title(f'Confusion Matrix ({best_name})')

# Misclassification analysis
wrong = X_test[preds != y_test]
right = X_test[preds == y_test]
print(f'Correct: {len(right)}, Wrong: {len(wrong)}, Error rate: {len(wrong)/len(y_test):.4f}')

# Feature importance if available
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)
    imp.tail(15).plot.barh(ax=axes[1])
    axes[1].set_title('Feature Importance')
else:
    axes[1].text(0.5, 0.5, 'No feature importance', ha='center')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## Interpretation & Business Insight
CGPA is the strongest predictor, followed by GRE and TOEFL. Research experience provides a clear advantage.

## Limitations
- Small dataset (~500 rows)
- Binarization threshold is arbitrary
- Dataset from one specific source

## How to Improve
- Try different binarization thresholds
- Ensemble with probability calibration
- Add cross-validation

## Production Considerations
- Threshold should be tuned per institution
- Regular retraining as standards evolve
- Transparency about false positive/negative tradeoffs

## Common Mistakes
- Forgetting to drop the original chance column (leakage!)
- Not stratifying the split
- Using accuracy alone on imbalanced classes

## Mini Challenge
1. Try threshold at 0.7 — how does it change class balance?
2. Add polynomial features
3. Plot ROC curve for each model

## Final Summary
Classified admission outcomes by binarizing probability. Boosting models slightly outperform logistic regression.

In [ ]:
# --- Save Metrics ---
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))